In [ ]:
import os
import sounddevice as sd
import numpy as np
from faster_whisper import WhisperModel
import ollama
from IPython.display import clear_output
import time
from pydantic import BaseModel
import torch
from llama_cpp import Llama

torch.set_num_threads(1)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from buffer import AudioBuffer
from transcription_pipeline import TranscriptionPipeline
from mergestrat import MergeStrategySelector

In [ ]:
print(sd.query_devices())

In [ ]:
model_size = "small"

In [ ]:
model_vad, utils = torch.hub.load(
    repo_or_dir= 'snakers4/silero-vad',
    model = 'silero_vad')

In [ ]:
llm_model = Llama(
    model_path="old_generated_files/llama-3.2-1b-instruct_best_working.Q8_0.gguf",
    n_gpu_layers=-1,
    n_ctx=2048,
    verbose=False
)

In [ ]:
whisper_model = WhisperModel(model_size, device="cuda")
audio_buffer = AudioBuffer(window_size = 7, overlap_cut_window = [3,4], end_cut_window = [6,7], model_vad = model_vad, utils = utils)
merge_selector = MergeStrategySelector(context_words = 10, model = llm_model)
pipeline = TranscriptionPipeline(whisper_model, merge_selector)

def audio_callback(indata, frames, times, status):
    audio_buffer.append_chunk(indata)
    
with sd.InputStream(samplerate = 16000, channels = 1, callback = audio_callback, blocksize=16000):
    while True:
        sd.sleep(100)
        window = audio_buffer.get_window()
        if window is None:
            continue
        transcribed_text = pipeline.process_and_merge(window)
        print(audio_buffer.get_buffer_size())
        print(transcribed_text)

In [ ]:
test2